In [12]:
import pandas as pd

from tqdm.auto import tqdm
from dotenv import load_dotenv

In [13]:
load_dotenv()

True

In [14]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [19]:
from src.rag import rag

In [20]:
df_question = pd.read_csv(
    "../data/processed/ground-truth-retrieval.csv"
)

ground_truth = df_question.to_dict(orient="records")

In [18]:
from src.ingest import ingest

# Load recipes and search index
df_recipes, index = ingest()

top_k_values = [3, 5, 10]

retrieval_results = []

for k in top_k_values:

    total = 0

    for row in ground_truth:

        results = index.search(
            query=row["question"],
            num_results=k
        )

        total += len(results)

    retrieval_results.append({
        "top_k": k,
        "average_results": total / len(ground_truth)
    })

retrieval_results

[{'top_k': 3, 'average_results': 3.0},
 {'top_k': 5, 'average_results': 5.0},
 {'top_k': 10, 'average_results': 10.0}]

In [21]:
import pandas as pd

retrieval_df = pd.DataFrame(retrieval_results)

retrieval_df

,top_k,average_results
0,3,3.0
1,5,5.0
2,10,10.0


In [22]:
best_k = retrieval_df.sort_values(
    "average_results",
    ascending=False
).iloc[0]["top_k"]

print(f"Selected Top-{int(best_k)} for the final system.")

Selected Top-10 for the final system.


In [32]:
df_question.columns

Index(['id', 'question'], dtype='str')

In [33]:
df_question.head()

,id,question
0,59957,Do I need to thaw and drain the frozen spinach...
1,59957,Can I substitute fresh lump crabmeat for the f...
2,68281,Can I use pork tenderloin instead of beef in t...
3,68281,What can I substitute if I cannot find scotch ...
4,71122,How do I know when the artichokes are fully co...


In [34]:
from src.ingest import ingest

df_recipes, index = ingest()

top_k_scores = []

for k in [3, 5, 10]:

    hits = 0

    for row in ground_truth:

        results = index.search(
            query=row["question"],
            num_results=k
        )

        retrieved_ids = [r["id"] for r in results]

        if row["id"] in retrieved_ids:
            hits += 1

    hit_rate = hits / len(ground_truth)

    top_k_scores.append({
        "top_k": k,
        "hit_rate": hit_rate
    })

top_k_scores

[{'top_k': 3, 'hit_rate': 0.05},
 {'top_k': 5, 'hit_rate': 0.05},
 {'top_k': 10, 'hit_rate': 0.2}]

In [23]:
import importlib

import src.llm
import src.rag

importlib.reload(src.llm)
importlib.reload(src.rag)

from src.rag import rag

In [24]:
rag("hello")

Using model: gemini-flash-latest


{'original_query': 'hello',
 'rewritten_query': 'What are some easy recipes for beginners?',
 'answer': "Hello! As your AI Kitchen Assistant, I can help you find a recipe based on your needs. Since you didn't specify a preference, here are the options available in our database:\n\n**If you want something quick (15 minutes or less):**\n*   **Sunny Side Up Eggs:** A very simple breakfast option using butter, eggs, salt, and pepper.\n*   **Southern Country Fried Steaks and Gravy:** A hearty comfort food dish featuring cube steaks, eggs, flour, and milk.\n\n**If you are looking for a snack or bread:**\n*   **Simple Biscuits (Lactose/Egg Free):** Great for beginners or kids; uses flour, baking powder, salt, margarine, and goat's milk.\n*   **Quick Cheese and Pepper Bread:** A savory quick bread using bread flour, sharp cheddar, and buttermilk; best for pairing with soup or chili.\n\n**If you want a main dish:**\n*   **Pecan Crusted Tilapia:** A simple and fresh seafood dish that takes about

In [25]:
import time

answers = []

for q in tqdm(ground_truth):
    answer = rag(q["question"])

    answers.append({
        "id": q["id"],
        "question": q["question"],
        "answer": answer
    })

    time.sleep(5)

  0%|          | 0/20 [00:00<?, ?it/s]

Using model: gemini-flash-latest
Using model: gemini-flash-latest
Using model: gemini-flash-latest
Using model: gemini-flash-latest
Using model: gemini-flash-latest
Using model: gemini-flash-latest
Using model: gemini-flash-latest
Using model: gemini-flash-latest
Using model: gemini-flash-latest
Using model: gemini-flash-latest
Using model: gemini-flash-latest
Using model: gemini-flash-latest
Using model: gemini-flash-latest
Using model: gemini-flash-latest
Using model: gemini-flash-latest
Using model: gemini-flash-latest
Using model: gemini-flash-latest
Using model: gemini-flash-latest
Using model: gemini-flash-latest
Using model: gemini-flash-latest


In [26]:
df_answers = pd.DataFrame(answers)

df_answers.head()

,id,question,answer
0,59957,Do I need to thaw and drain the frozen spinach...,{'original_query': 'Do I need to thaw and drai...
1,59957,Can I substitute fresh lump crabmeat for the f...,{'original_query': 'Can I substitute fresh lum...
2,68281,Can I use pork tenderloin instead of beef in t...,{'original_query': 'Can I use pork tenderloin ...
3,68281,What can I substitute if I cannot find scotch ...,{'original_query': 'What can I substitute if I...
4,71122,How do I know when the artichokes are fully co...,{'original_query': 'How do I know when the art...


In [27]:
df_answers.to_csv(
    "../data/processed/rag-answers.csv",
    index=False
)

In [28]:
df_answers = pd.read_csv(
    "../data/processed/rag-answers.csv"
)

df_answers.head()

,id,question,answer
0,59957,Do I need to thaw and drain the frozen spinach...,{'original_query': 'Do I need to thaw and drai...
1,59957,Can I substitute fresh lump crabmeat for the f...,{'original_query': 'Can I substitute fresh lum...
2,68281,Can I use pork tenderloin instead of beef in t...,{'original_query': 'Can I use pork tenderloin ...
3,68281,What can I substitute if I cannot find scotch ...,{'original_query': 'What can I substitute if I...
4,71122,How do I know when the artichokes are fully co...,{'original_query': 'How do I know when the art...


In [29]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [30]:
judge_prompt_template = """
You are an expert cooking assistant.

You will receive:

Question:
{question}

Answer:
{answer}

Evaluate whether the answer correctly answers the user's question.

Return ONLY one word:

RELEVANT
or
NON_RELEVANT
""".strip()

In [31]:
judge_prompt_template_v2 = """
You are an expert recipe evaluator.

Evaluate whether the generated answer correctly, completely, and accurately answers the user's question using the retrieved recipe information.

Evaluation Rules:

- RELEVANT:
  - The answer directly answers the user's question.
  - The answer is factually correct.
  - The answer contains enough useful information.

- NON_RELEVANT:
  - The answer is incorrect.
  - The answer is incomplete.
  - The answer is unrelated to the question.

Question:
{question}

Answer:
{answer}

Return ONLY one word:

RELEVANT
or
NON_RELEVANT
""".strip()

In [32]:
def llm_judge(prompt):

    response = client.models.generate_content(
       model="gemini-3.1-flash-lite",
        contents=prompt
    )

    return response.text.strip()

In [33]:
def llm_judge_v2(prompt):

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )

    return response.text.strip()

In [34]:
sample_answers = df_answers.sample(
    n=10,
    random_state=42
)

sample_answers

,id,question,answer
0,59957,Do I need to thaw and drain the frozen spinach...,{'original_query': 'Do I need to thaw and drai...
17,159716,Is it necessary to refrigerate the spread for ...,{'original_query': 'Is it necessary to refrige...
15,52698,What are the recommended measurements or ratio...,{'original_query': 'What are the recommended m...
1,59957,Can I substitute fresh lump crabmeat for the f...,{'original_query': 'Can I substitute fresh lum...
8,136941,Can I substitute the liquid smoke if I don't h...,"{'original_query': ""Can I substitute the liqui..."
5,71122,What is the best way to prepare and remove the...,{'original_query': 'What is the best way to pr...
11,31958,How much confectioners' sugar and water should...,"{'original_query': ""How much confectioners' su..."
3,68281,What can I substitute if I cannot find scotch ...,{'original_query': 'What can I substitute if I...
18,64215,"Can I add chocolate chips to this recipe, and ...",{'original_query': 'Can I add chocolate chips ...
16,159716,Can I use a different type of meat besides chi...,{'original_query': 'Can I use a different type...


In [35]:
row = df_answers.iloc[0]

prompt = judge_prompt_template.format(
    question=row["question"],
    answer=row["answer"]
)

print(llm_judge(prompt))

RELEVANT


In [36]:
import time

judgements = []

for _, row in tqdm(df_answers.iterrows(), total=len(df_answers)):

    prompt = judge_prompt_template.format(
        question=row["question"],
        answer=row["answer"]
    )

    judgement = llm_judge(prompt)

    judgements.append({
        "id": row["id"],
        "question": row["question"],
        "judgement": judgement
    })

    time.sleep(5)

  0%|          | 0/20 [00:00<?, ?it/s]

In [37]:
import time

judgements_v2 = []

for _, row in tqdm(sample_answers.iterrows(), total=len(sample_answers)):

    prompt = judge_prompt_template_v2.format(
        question=row["question"],
        answer=row["answer"]
    )

    judgement = llm_judge_v2(prompt)

    judgements_v2.append({
        "id": row["id"],
        "question": row["question"],
        "judgement": judgement
    })

    time.sleep(5)


  0%|          | 0/10 [00:00<?, ?it/s]

In [38]:
df_judgements = pd.DataFrame(judgements)

df_judgements

,id,question,judgement
0,59957,Do I need to thaw and drain the frozen spinach...,RELEVANT
1,59957,Can I substitute fresh lump crabmeat for the f...,RELEVANT
2,68281,Can I use pork tenderloin instead of beef in t...,RELEVANT
3,68281,What can I substitute if I cannot find scotch ...,RELEVANT
4,71122,How do I know when the artichokes are fully co...,RELEVANT
5,71122,What is the best way to prepare and remove the...,RELEVANT
6,163767,Can I substitute the dry red wine with somethi...,RELEVANT
7,163767,If I want to cook the pork to a lower internal...,NON_RELEVANT
8,136941,Can I substitute the liquid smoke if I don't h...,RELEVANT
9,136941,How long should I store this sauce in the refr...,NON_RELEVANT


In [39]:
df_judgements_v2 = pd.DataFrame(judgements_v2)

In [40]:
df_judgements.to_csv(
    "../data/processed/judgements.csv",
    index=False
)

In [41]:
df_judgements_v2.to_csv(
    "../data/processed/judgements_v2.csv",
    index=False
)

In [42]:
score = (
    df_judgements["judgement"]
    .str.upper()
    .eq("RELEVANT")
    .mean()
)

print(f"RAG Score: {score:.2%}")

RAG Score: 85.00%


In [43]:
score_v2 = (
    df_judgements_v2["judgement"]
    .str.upper()
    .eq("RELEVANT")
    .mean()
)

print(f"Judge V2 Score: {score_v2:.2%}")

Judge V2 Score: 100.00%


In [44]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [45]:
from src.rag import rag

result = rag("healthy breakfast")
print(result)

Using model: gemini-flash-latest
{'original_query': 'healthy breakfast', 'rewritten_query': 'nutritious and healthy breakfast recipes', 'answer': 'For a healthy breakfast, I recommend the following options based on your preference for a meal or a quick drink:\n\n**Meal Option:**\n*   **Spicy Cheesy BLT Egg Wrap:** A nutritious and easy-to-prepare meal using whole wheat wraps, eggs, turkey bacon, and fresh vegetables. It takes 14 minutes to make and can be served with fruit salad or cottage cheese.\n\n**Smoothie/Drink Options:**\n*   **Orange and Mango Breakfast Smoothie:** Specifically tagged as "healthy," this smoothie is thick and nutritious. It takes 6 minutes to prepare and is low in calories, sodium, and cholesterol.\n*   **Breakfast Smoothie Quickie:** A convenient, grab-and-go option made with strawberry-banana yogurt, orange juice, and a frozen banana. It takes only 2 minutes to prepare.\n*   **Diabetic Breakfast Drink:** A quick, nutritious option using milk, banana, egg, whea